# map/ — Colab SWEEP `min_dist` × `n_neighbors` (Parametric UMAP)

Quét nhiều cấu hình pumap để chọn map **dễ nhìn nhất**, rồi mới chạy `run_colab.ipynb` (production) với config đã chọn.

Khác production: **KHÔNG lưu reducer** (reducer = để serve 'you are here', sweep chỉ cần ngắm) → mỗi config chỉ fit → coords → 1 HTML. Cùng dùng `cluster` 128-d (fit 1 lần) để **màu cụm giống nhau giữa mọi map** → so sánh công bằng.

Output ghi `map/outputs/sweep/pumap_n<nn>_d<md>.html` (subfolder riêng, KHÔNG đụng `pumap2d_demo.html` production). Kéo cả folder `sweep/` về local, mở từng HTML so.

> Data đặt sẵn `MyDrive/map_data/` như production (`artifacts/` + `cleaned-data/details.csv`).

In [ ]:
# 0) Mount Drive + clone code + cd
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

MAP_DATA = '/content/drive/MyDrive/map_data'
assert os.path.isdir(MAP_DATA), f'Khong thay {MAP_DATA}'

REPO   = 'https://github.com/CrocodixD/anime-recommender.git'
BRANCH = 'main'      # ⚠ map/ nam o main (anime_map da cu, DUNG dung)
CODE   = '/content/recommender'

if os.path.exists(CODE):
    !cd {CODE} && git fetch origin && git checkout {BRANCH} && git pull
else:
    !git clone -b {BRANCH} {REPO} {CODE}

%cd {CODE}

In [ ]:
# 1) Symlink data Drive -> path repo (giong production)
import os

assert os.path.isdir(f'{CODE}/map'), f'Khong thay {CODE}/map trong clone.'
os.makedirs(f'{MAP_DATA}/outputs', exist_ok=True)
links = {
    f'{MAP_DATA}/artifacts':    f'{CODE}/artifacts',
    f'{MAP_DATA}/cleaned-data': f'{CODE}/cleaned-data',
    f'{MAP_DATA}/outputs':      f'{CODE}/map/outputs',
}
for src, dst in links.items():
    assert os.path.isdir(src), f'Thieu {src} tren Drive'
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if os.path.islink(dst):
        os.unlink(dst)
    elif os.path.exists(dst):
        raise SystemExit(f'{dst} da ton tai (khong phai symlink) — xoa tay roi chay lai.')
    os.symlink(src, dst)
    print('link', dst, '->', src)

In [ ]:
# 2) Cai lib + EP TensorFlow chay CPU
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
!pip install -q umap-learn
import tensorflow as tf
print('tf', tf.__version__, '| GPU TF:', tf.config.list_physical_devices('GPU'))

In [ ]:
# 3) Base + cluster 128-d — chay 1 LAN (mau cum dung chung cho moi config sweep)
!python map/build_base.py
!python map/cluster.py --algo kmeans --k 20

In [ ]:
# 4) SWEEP: fit pumap moi (n_neighbors, min_dist) -> HTML mau theo cum (KHONG luu reducer)
import os, sys, time, gc
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
sys.path.insert(0, 'map')                 # de import _common / project / viz
import pandas as pd
import _common as C
from project import fit
from viz import load_plot_df, build_fig

# --- luoi quet (sua thoai mai). min_dist nho + n_neighbors nho = cum chat, nhieu khoang trang ---
N_NEIGHBORS = [10, 15, 30, 50]
MIN_DIST    = [0.0, 0.1, 0.5]

base, X = C.load_base()
idx = base['anime_idx'].to_numpy()
sweep_dir = C.OUTPUTS / 'sweep'
sweep_dir.mkdir(parents=True, exist_ok=True)

total = len(N_NEIGHBORS) * len(MIN_DIST)
k = 0
for nn in N_NEIGHBORS:
    for md in MIN_DIST:
        k += 1
        t0 = time.time()
        reducer, emb = fit(X, nn, md)
        coords = pd.DataFrame({'anime_idx': idx,
                               'x': emb[:, 0].astype('float32'),
                               'y': emb[:, 1].astype('float32')})
        df = load_plot_df('pumap2d', 'cluster', 'kmeans', coords=coords)
        fig = build_fig(df, 'cluster', f'pumap n_neighbors={nn} min_dist={md} · {len(df):,}')
        out = sweep_dir / f'pumap_n{nn}_d{md}.html'
        fig.write_html(out, include_plotlyjs=True, full_html=True)
        del reducer, emb, coords, df, fig; gc.collect()
        print(f'[{k}/{total}] [{time.time()-t0:.0f}s] -> {out.name}')
print('XONG. HTML o map/outputs/sweep/ (da ghi ve Drive).')

In [ ]:
# 5) Liet ke HTML sweep tren Drive (keo ca folder ve local mo browser)
!ls -lhS /content/drive/MyDrive/map_data/outputs/sweep/*.html